# agentic-verifier-selfrefine — full run, from scratch

Run every cell in order, top to bottom. Nothing from an earlier session is needed or
used: the zip carries the code and datasets, and every result below is produced here.

Each cell either succeeds or stops the run. Four of them exist only to stop it early:

- **7, smoke test.** Trains, merges, saves, loads and generates once at the smallest
  size, about four minutes, so a library incompatibility shows up now and not two
  hours in.
- **8, split gate.** Refuses to continue if the training data overlaps the benchmark.
- **9, clean slate.** Refuses to continue if outputs or models from another run are
  already on disk.
- **16, selection gate.** Refuses to continue if no working checkpoint was found.

| section | what | hardware | time |
|---|---|---|---|
| 1–6 | upload, pinned install, tests | CPU | ~6 min |
| 7 | smoke test | GPU | ~4 min |
| 8–10 | gates, datasets, synthetic experiment | CPU | ~4 min |
| 11–12 | four training passes + summary | GPU | ~60 min |
| 13–14 | agent loop + five-condition ladder, on the last pass's model | GPU | ~75 min |
| 15–16 | select a checkpoint that did not collapse, score it | GPU | ~15–55 min |
| 17–18 | agent loop + ladder again, on the selected model | GPU | ~75 min |
| 19 | final download | – | – |

About four to five hours on a T4. A backup zip is downloaded after every GPU stage, so
a dropped session costs at most the stage that was running. See the end for resuming.

Runtime → Change runtime type → T4 GPU before starting.


## 1. Check the GPU


In [ ]:
!nvidia-smi


## 2. Upload the project zip

Earlier uploads are removed first. Colab saves a second upload of the same filename as
`name (1).zip`, and unpacking would then quietly reopen the old one.


In [ ]:
import glob, os
from google.colab import files

for z in glob.glob('/content/agentic-verifier-selfrefine*.zip'):
    os.remove(z)

uploaded = files.upload()          # pick agentic-verifier-selfrefine.zip
print('uploaded:', list(uploaded))


## 3. Unpack


In [ ]:
import glob, os, shutil

zips = sorted(glob.glob('/content/agentic-verifier-selfrefine*.zip'), key=os.path.getmtime)
assert zips, 'no zip found -- run cell 2 first'
ZIP_PATH = zips[-1]
print('unpacking', ZIP_PATH)

shutil.rmtree('/content/agentic-verifier-selfrefine', ignore_errors=True)
!unzip -q -o "{ZIP_PATH}" -d /content
os.chdir('/content/agentic-verifier-selfrefine')
for needed in ('src/smoke_test.py', 'src/select_checkpoint.py', 'requirements-colab.txt'):
    assert os.path.exists(needed), f'{needed} missing -- this is an older zip'
!ls


## 4. Install the pinned GPU stack

`requirements-colab.txt` pins `trl`, `peft` and `transformers` to the versions the
recorded runs used. An unpinned upgrade installs whatever is newest today, and the
training scripts sit directly on APIs that change between releases.

`torch` is left alone on purpose: Colab ships a build matched to its own CUDA runtime,
and replacing it is the install most likely to break the GPU.

`torchao` is removed, not upgraded. Colab preinstalls a version `peft` rejects, nothing
here uses it, and its newer wheels are compiled against one specific `torch`.


In [ ]:
!pip -q uninstall -y torchao
!pip -q install -r /content/agentic-verifier-selfrefine/requirements-colab.txt -r /content/agentic-verifier-selfrefine/requirements-dev.txt
assert globals().get('_exit_code', 0) == 0, 'pip install failed -- stop and send the output above'
!pip freeze > /content/agentic-verifier-selfrefine/results/raw/pip_freeze.txt
!python -c "import torch, transformers, trl, peft, accelerate, datasets; print('torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__, '| accelerate', accelerate.__version__, '| datasets', datasets.__version__)"
assert globals().get('_exit_code', 0) == 0, 'the GPU stack does not import -- stop and send the output above'


## 5. Helpers: stop on failure, back up after every stage

`check()` stops the run when the shell command above it failed. Without it, a failed
stage is just red text and the next cell starts anyway.

`backup()` zips `results/` **without the model weights** (under 1 MB) and copies it to
Google Drive, which survives a closed tab; Drive asks for permission once, here.

Drive is the default because browser downloads were unreliable in the recorded run:
the multiple-download prompt was blocked, and the workaround written mid-run zipped
`results/` *with* the weights (1.8 GB per backup), which is too large to retrieve.
Set `DOWNLOAD = True` only if you also want each zip pushed to the browser.


In [ ]:
import os, shutil, subprocess
from google.colab import files

ROOT = '/content/agentic-verifier-selfrefine'
USE_DRIVE = True     # copy every backup to Drive (recommended; see above)
DOWNLOAD = False     # also push every backup to the browser

DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/agentic_backups'
    os.makedirs(DRIVE_DIR, exist_ok=True)

def check(stage):
    code = globals().get('_exit_code', 0)
    assert code == 0, f'{stage} FAILED (exit {code}). Stop here and send the output above.'
    print(f'OK: {stage}')

def backup(tag):
    os.makedirs('/content/out', exist_ok=True)
    path = f'/content/out/results_{tag}.zip'
    if os.path.exists(path):
        os.remove(path)
    subprocess.run(['zip', '-rq', path, 'results', '-x', 'results/sft-merged-model/*',
                    'results/lora-*/*', 'results/selected-checkpoint/*'], cwd=ROOT, check=True)
    print(f'backup: {path} ({os.path.getsize(path) / 1e6:.1f} MB)')
    size_mb = os.path.getsize(path) / 1e6
    # Never ship weights in a backup: a zip this large cannot be fetched back.
    assert size_mb < 50, f'{path} is {size_mb:.0f} MB -- model weights leaked into the backup'
    if DRIVE_DIR:
        shutil.copy(path, DRIVE_DIR)
        print('  copied to Drive:', DRIVE_DIR)
    if DOWNLOAD or not DRIVE_DIR:
        files.download(path)

print('helpers ready | Drive backups:', 'on' if DRIVE_DIR else 'off')


**Only when resuming in a new session:** set `RESTORE = True` and pick the newest
`results_*.zip` you downloaded. It is unpacked over `results/`, so the final zip still
holds everything. Leave it `False` on a first run.


In [ ]:
RESTORE = False
if RESTORE:
    up = files.upload()
    name = os.path.abspath(list(up)[0])
    subprocess.run(['unzip', '-q', '-o', name, '-d', ROOT], check=True)
    print('restored', name)
    print(sorted(os.listdir(os.path.join(ROOT, 'results', 'raw'))))


## 6. Run the test suite

Must be green before anything is trained. It checks the verifier's verdicts and
accepted answer formats, cross-process determinism, the experiment's own validity
conditions, the checkpoint-selection rule, and that the datasets do not overlap the
benchmark.


In [ ]:
!cd /content/agentic-verifier-selfrefine && python -m pytest
check('test suite')


## 7. Smoke test: the whole chain once, as small as it goes

Runs the real training scripts for one epoch each, merges, saves, loads both models
back, generates, and grades. About four minutes. Its answers come from an
under-trained model and say nothing about quality; the only question is whether every
library call in the chain still works. It removes the checkpoints it made.

**If this fails, do not continue.** Send the output.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python smoke_test.py
check('smoke test')


## 8. Gate: the split must be disjoint

A different seed is not a disjoint sample. This measures the overlap by equation,
which is the only comparison that means anything, and stops if it is not zero.

Do not compare by `problem_id`. Ids are positional within a draw, so the benchmark and
the training draw share `linear_0000`..`linear_0014` while none of them is the same
equation.


In [ ]:
import sys
sys.path.insert(0, '/content/agentic-verifier-selfrefine/src')
import splits
from tasks import contamination

held = splits.held_out()
for n, name in [(100, 'SFT draw'), (60, 'DPO draw')]:
    report = contamination(splits.training(n), held)
    print(f"{name:10} {report['n_overlapping']}/{report['n_held_out']} overlapping equations")
    assert report['n_overlapping'] == 0, 'split is contaminated; stop here'
assert contamination(splits.validation(), held)['n_overlapping'] == 0

print(f'\nbenchmark: {len(held)} problems, seed {splits.HELD_OUT_SEED}')
print('GATE PASSED — training and validation are disjoint from the benchmark')


## 9. Gate: clean slate

Every result of this run must come from this run. The zip ships no model weights and
no outputs at the paths the stages write to; this confirms it, so that nothing below
can be an old file read as a new one. On a resumed session (`RESTORE = True` above) the
restored outputs are listed and allowed.


In [ ]:
import glob, os

models = [p for p in ('results/sft-merged-model', 'results/lora-sft-adapter',
                      'results/lora-dpo-zeroshot-adapter', 'results/selected-checkpoint')
          if os.path.exists(os.path.join(ROOT, p))]
assert not models, (f'model weights already on disk: {models}. At the START of a run: unpack a '
                    'fresh zip (cell 3). In the MIDDLE of a run: skip this cell, it has already passed.')

outputs = sorted(os.path.relpath(p, ROOT) for pattern in (
    'results/raw/eval/*', 'results/llm_agent_experiment*',
    'results/feedback_protocol_ablation*', 'results/raw/checkpoint_selection.json')
    for p in glob.glob(os.path.join(ROOT, pattern)))
if globals().get('RESTORE', False):
    print(f'resumed session: {len(outputs)} restored output files kept')
else:
    assert not outputs, f'outputs from another run are present: {outputs[:5]}'
print('GATE PASSED — nothing stale on disk')


## 10. Datasets and the synthetic experiment (CPU)

Rebuilds both training files and runs the CPU turn-budget experiment. The builders
print the contamination as they write. Confirm `0/45` in the output.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py data cpu
check('datasets + CPU experiment')


## 11. Four training passes

`sft dpo eval`, never `eval` alone. Decoding is greedy and therefore deterministic, so
repeating the evaluation on one checkpoint prints the same table four times and
measures nothing. Each pass here trains a fresh model, and the spread between passes
is the GPU non-determinism, which is the quantity in question: one pass is a draw, not
a measurement.

About 60 minutes.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py sft dpo eval --repeat=4
backup('1_after_passes')
check('four training passes')


## 12. Combine the passes

Never read one pass on its own. This prints every pass, the spread, the DPO delta per
pass, the verifier's reason for each failure, and which problems move between passes.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python summarize_eval.py
check('summary of the passes')


## 13. The agent loop, on the last pass's model

Verifier feedback against a re-sampling control, paired under identical sampling
seeds. About 25 minutes.

This runs on whatever model pass 4 left behind, which may be a collapsed one. That is
intended. Sections 17–18 repeat both experiments on a model known to work, and the
pair shows whether the benefit of feedback depends on the model being damaged.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py agent
backup('2_after_first_agent')
check('agent loop on the last-pass model')


## 14. The five-condition feedback ladder, on the same model

`retry` / `generic` / `diagnostic` / `actionable` / `constrained`. Separates “feedback
does not help” from “feedback was delivered badly”. About 50 minutes.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py ablation
backup('3_after_first_ladder')
check('feedback ladder on the last-pass model')


## 15. Select a checkpoint that did not collapse

The DPO stage can wreck the model it is given. This retrains until it has not: the
SFT+DPO rate must be within three problems of the same attempt's SFT-only rate, both
measured on a **validation** set of 45 problems that shares nothing with the benchmark.
The benchmark is never consulted, because retraining until the test score looks good
is selection on the test set.

Every attempt is logged, not only the survivor. The number of attempts needed is a
direct estimate of the collapse rate, which is a finding in its own right.

About 8 minutes per attempt, at most six attempts.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py select
backup('4_after_selection')
# The selected weights (~1 GB) are copied to Drive ONCE, separately from the small
# result zips, so a later session can re-run the agent loop or the ladder on exactly
# this checkpoint (notebooks/colab_followup.ipynb does that).
if DRIVE_DIR:
    dst = os.path.join(DRIVE_DIR, 'selected-checkpoint')
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(os.path.join(ROOT, 'results/selected-checkpoint'), dst)
    print('selected checkpoint copied to', dst)


## 16. Gate, then score the selected checkpoint once

Stops the run if selection kept nothing, because the model on disk would then be a
collapsed one and everything below would be mislabelled. It also moves the outputs of
sections 13–14 into `results/raw/before_selection/`: the agent loop and the ladder
write to fixed filenames, and the re-run would otherwise overwrite them.


In [ ]:
import glob, json, os, shutil

log_path = os.path.join(ROOT, 'results/raw/checkpoint_selection.json')
assert os.path.exists(log_path), 'section 15 did not finish; run it first'
log = json.load(open(log_path))
for a in log['attempts']:
    print(f"attempt {a['attempt']}: SFT {a['sft_validation']:.3f} | SFT+DPO "
          f"{a['sft_dpo_validation']:.3f} | {'collapsed' if a['collapsed'] else 'kept'}")
assert log['kept'] is not None, (
    'no checkpoint survived selection -- the model on disk is a collapsed one; stop here')
print(f"\n{log['n_collapsed']} of {log['n_attempts']} attempts collapsed")

keep = os.path.join(ROOT, 'results/raw/before_selection')
os.makedirs(keep, exist_ok=True)
moved = []
for pattern in ('results/llm_agent_experiment*', 'results/feedback_protocol_ablation*'):
    for path in glob.glob(os.path.join(ROOT, pattern)):
        shutil.move(path, os.path.join(keep, os.path.basename(path)))
        moved.append(os.path.basename(path))
print(f'moved {len(moved)} earlier output files to results/raw/before_selection/')
print('GATE PASSED — a working checkpoint is in place')


The selected checkpoint's score goes to `results/raw/selected_eval/`, apart from the
four passes in `results/raw/eval/`. A chosen model is not a fifth draw from the same
distribution, and the two must never be averaged together.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && AVS_EVAL_DIR=../results/raw/selected_eval python rerun_all.py eval && python summarize_eval.py --dir ../results/raw/selected_eval
backup('5_after_selected_eval')
check('benchmark score of the selected checkpoint')


## 17. The agent loop, on the selected checkpoint

If the benefit of feedback survives on a model that works, the central claim stands
without its largest caveat. If it shrinks or disappears, that is equally worth knowing.


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py agent
backup('6_after_selected_agent')
check('agent loop on the selected checkpoint')


## 18. The feedback ladder, on the selected checkpoint


In [ ]:
!cd /content/agentic-verifier-selfrefine/src && python rerun_all.py ablation
backup('7_after_selected_ladder')
check('feedback ladder on the selected checkpoint')


## 19. Final download

One zip with everything this session produced: the four passes, the first agent and
ladder runs (`raw/before_selection/`), the selection log, the selected checkpoint's
benchmark score, both experiments re-run on it, a run manifest per stage, the smoke
test record and the installed package list.


In [ ]:
backup('final')


---

### What to send back

`results_final.zip`, and this notebook with its outputs (File → Download → Download
.ipynb). The numbers are **not** promoted into `results/recorded/manifest.json`
automatically. Deciding that a number is now the project's result is a judgement made
after looking at it, not a side effect of the script that produced it.

### If the session is cut off

The Colab disk is lost with the session, the downloaded backups are not. In a new
session run cells 1–5, set `RESTORE = True` and pick the newest backup, run 6–9, then:

| cut off during | continue with |
|---|---|
| 11 (passes) | 11 again; passes accumulate and 12 summarises all of them |
| 13 or 14 | 15. The last pass's model is gone, so 13–14 are skipped; 17–18 still run |
| 15 to 18 | 15 again. Selection retrains, so it needs nothing from before |

### If a cell fails

| symptom | cause |
|---|---|
| `SMOKE TEST FAILED at: ...` | a library call no longer works on this stack; send the output |
| `pip install failed` | a pinned version could not be installed; send the output |
| `stage 'X' needs [...]` | an earlier stage did not run; the driver refuses to evaluate a missing or stale model |
| `--repeat=N was given, but none of [...] retrains` | use `sft dpo eval --repeat=N` |
| `split is contaminated` | the split gate failed; do not train |
| `outputs from another run are present` | the zip was not unpacked fresh; re-run cell 3 |
| `no checkpoint survived selection` | every attempt collapsed; stop and send the log |
| `older zip` | cell 2 received a previous version of the project zip |
| `torch.cuda.is_available() is False` | Runtime → Change runtime type → T4 GPU |
